# P2: Audit, Cleaning Log & Provenance Brief
**DSA 405 · Fall 2026 · Avery Atkinson · alatkin3**

## Provenance Brief

This project uses two public datasets about New York City. The first is a list of restaurant health inspections, published by the NYC Department of Health and Mental Hygiene on the city's Open Data website. Inspectors visit restaurants, record any food-safety problems they find, and give a score. The city publishes the results so the public can see how restaurants are doing. Each row is one problem found during one inspection, covering 2007 to October 2026. The second dataset is household income by zip code, produced by the U.S. Census Bureau from a yearly survey of a sample of households. These are typical (median) incomes for 2018 to 2022, so they are estimates, not exact counts. Together, they can show whether restaurants in lower-income areas tend to have more serious violations. They cannot show why, they say nothing about individual people, and they only describe restaurants that were inspected.

## 1. Loading the data
Both datasets were downloaded on Oct 8, 2026 and saved unchanged in the `raw` folder. All later work uses copies.

In [1]:
import pandas as pd, os
os.makedirs("raw", exist_ok=True)

url = "https://data.cityofnewyork.us/resource/43nn-pn8j.csv?$limit=500000"
insp = pd.read_csv(url, dtype=str)   # dtype=str keeps everything exactly as published
insp.to_csv("raw/inspections_raw.csv", index=False)
print(insp.shape)
insp.head()

(296507, 27)


,camis,dba,boro,building,street,zipcode,phone,cuisine_description,inspection_date,action,...,inspection_type,latitude,longitude,community_board,council_district,census_tract,bin,bbl,nta,location
0,50126972,MASALA CORP,Manhattan,1802,65TH ST,NaN,9174003090,NaN,1900-01-01T00:00:00.000,NaN,...,NaN,0,0,NaN,NaN,NaN,NaN,1,NaN,NaN
1,50189028,KOSTANTINA FROZEN YOGURT,Queens,72-38,AUSTIN STREET,11375,9177676656,NaN,1900-01-01T00:00:00.000,NaN,...,NaN,40.71874452827,-73.841335851082,406,29,073700,4077933,4032557501,QN17,POINT (-73.841335851082 40.71874452827)
2,50177141,REMYMIDTOWN LLC,Manhattan,35,EAST 13 STREET,10003,3474255641,NaN,1900-01-01T00:00:00.000,NaN,...,NaN,40.734449766582,-73.9920652807,102,02,006100,1009214,1005650005,MN23,POINT (-73.9920652807 40.734449766582)
3,40380154,KUMNESS INC.,Manhattan,2549,AMSTERDAM AVENUE,10033,2129239470,NaN,1900-01-01T00:00:00.000,NaN,...,NaN,40.85151325954,-73.928396286343,112,10,026700,1063584,1021490093,MN35,POINT (-73.928396286343 40.85151325954)
4,50171259,TM:RW CAFE,Manhattan,220,WEST 42 STREET,10036,9173551075,NaN,1900-01-01T00:00:00.000,NaN,...,NaN,40.756404749307,-73.987857559104,105,03,011300,1024663,1010130042,MN17,POINT (-73.987857559104 40.756404749307)


In [2]:
import requests
from google.colab import userdata

key = userdata.get("CENSUS_KEY")
url = "https://api.census.gov/data/2022/acs/acs5/subject"
params = {"get": "NAME,S1901_C01_012E",
          "for": "zip code tabulation area:*",
          "key": key}
resp = requests.get(url, params=params,
                    headers={"User-Agent": "DSA405-student-project (NC State)"})
print(resp.status_code, resp.headers.get("Content-Type"))

try:
    data = resp.json()
except ValueError:
    print("NOT DATA. The Census site sent this instead:\n", resp.text[:300])
    raise

income = pd.DataFrame(data[1:], columns=data[0])
income.to_csv("raw/census_income_raw.csv", index=False)
print(income.shape)
income.head()

200 application/json;charset=utf-8
(33774, 3)


,NAME,S1901_C01_012E,zip code tabulation area
0,ZCTA5 00601,17526,00601
1,ZCTA5 00602,20260,00602
2,ZCTA5 00603,17703,00603
3,ZCTA5 00606,19603,00606
4,ZCTA5 00610,22796,00610


## 2. Audit
The audit below checks every column for missing values and distinct values, then counts specific problems.

In [3]:
insp = pd.read_csv("raw/inspections_raw.csv", dtype=str)

audit = pd.DataFrame({
    "missing": insp.isna().sum(),
    "missing_pct": (insp.isna().mean() * 100).round(1),
    "distinct": insp.nunique(),
    "example": insp.apply(lambda c: c.dropna().iloc[0] if c.notna().any() else None),
})
print(insp.shape)
audit

(296507, 27)


,missing,missing_pct,distinct,example
camis,0,0.0,31509,50126972
dba,3,0.0,24665,MASALA CORP
boro,0,0.0,6,Manhattan
building,1010,0.3,8568,1802
street,52,0.0,2224,65TH ST
zipcode,3108,1.0,254,11375
phone,104,0.0,27824,9174003090
cuisine_description,3958,1.3,90,Latin American
inspection_date,0,0.0,1970,1900-01-01T00:00:00.000
action,3878,1.3,5,No violations were recorded at the time of thi...


In [4]:
lat = pd.to_numeric(insp.latitude, errors="coerce")
lon = pd.to_numeric(insp.longitude, errors="coerce")
score = pd.to_numeric(insp.score, errors="coerce")
date = pd.to_datetime(insp.inspection_date, errors="coerce")

print("placeholder 1900 dates:", (date.dt.year == 1900).sum())
print("missing zip codes:", insp.zipcode.isna().sum())
print("zip codes not 5 digits:", (insp.zipcode.dropna().str.len() != 5).sum())
print("latitude/longitude = 0:", ((lat == 0) | (lon == 0)).sum())
print("exact duplicate rows:", insp.duplicated().sum())
print("score range:", score.min(), "to", score.max())
print("real date range:", date[date.dt.year > 1900].min(), "to", date.max())
print()
print(insp.critical_flag.value_counts(dropna=False))
print(insp.boro.value_counts(dropna=False))

placeholder 1900 dates: 3878
missing zip codes: 3108
zip codes not 5 digits: 0
latitude/longitude = 0: 3047
exact duplicate rows: 192
score range: 0.0 to 214.0
real date range: 2007-08-10 09:06:00 to 2026-10-07 00:00:00

critical_flag
Critical          155307
Not Critical      132541
Not Applicable      8659
Name: count, dtype: int64
boro
Manhattan        109660
Brooklyn          75003
Queens            74446
Bronx             27528
Staten Island      9521
0                   349
Name: count, dtype: int64


In [5]:
income = pd.read_csv("raw/census_income_raw.csv", dtype=str)
inc = pd.to_numeric(income["S1901_C01_012E"], errors="coerce")

print("rows:", len(income), "| unique zip areas:", income["zip code tabulation area"].nunique())
print("not a number:", inc.isna().sum())
print("negative codes (not real incomes):", (inc < 0).sum())
print(income.loc[inc < 0, "S1901_C01_012E"].value_counts())
print("real income range:")
print(inc[inc > 0].describe())

rows: 33774 | unique zip areas: 33774
not a number: 0
negative codes (not real incomes): 3153
S1901_C01_012E
-666666666    3153
Name: count, dtype: int64
real income range:
count     30621.000000
mean      73132.626368
std       31359.582064
min        2499.000000
25%       53466.000000
50%       66993.000000
75%       85313.000000
max      250001.000000
Name: S1901_C01_012E, dtype: float64


In [6]:
year = date.dt.year
print(pd.crosstab(insp.critical_flag,
                  [year == 1900, insp.violation_code.isna()],
                  rownames=["critical_flag"],
                  colnames=["placeholder_date", "no_violation_code"]))
print()
ph = year == 1900
print("boro '0' rows:", (insp.boro == "0").sum(), "| of those, placeholder dates:", ((insp.boro == "0") & ph).sum())
print("zero-coordinate rows that are placeholders:", (((lat == 0) | (lon == 0)) & ph).sum())
print("missing-zip rows that are placeholders:", (insp.zipcode.isna() & ph).sum())
print()
key = insp.camis + "|" + insp.inspection_date
tmp = pd.DataFrame({"k": key, "s": score})[year > 1900]
print("real inspections:", tmp.k.nunique())
print("inspections with more than one distinct score:", (tmp.groupby("k").s.nunique() > 1).sum())

placeholder_date    False       True 
no_violation_code   False True  True 
critical_flag                        
Critical           155307     0     0
Not Applicable       1830  2951  3878
Not Critical       132541     0     0

boro '0' rows: 349 | of those, placeholder dates: 68
zero-coordinate rows that are placeholders: 86
missing-zip rows that are placeholders: 136

real inspections: 84332
inspections with more than one distinct score: 2


In [7]:
odd = insp[(insp.critical_flag == "Not Applicable") & (year > 1900) & insp.violation_code.notna()]
print(odd.violation_description.value_counts().head(8))
print(odd.inspection_type.value_counts().head(5))

violation_description
The original nutrition fact labels or ingredient label for a cooking oil, shortening or margarine or food item sold in bulk, or acceptable manufacturer’s documentation not maintained on site.                                                                                                                                                                             460
Caloric content not posted on menus, menu boards or food tags, in a food service establishment that is 1 of 15 or more outlets operating the same type of business nationally under common ownership or control, or as a franchise or doing business under the same name, for each menu item that is served in portions, the size and content of which are standardized.    289
Sign prohibiting smoking or using electronic cigarettes not conspicuously posted.                                                                                                                                                                 

### Problems found (counts from the audit)

| Dataset | Problem | Count |
|---|---|---|
| Inspections | Placeholder inspection date 1900-01-01 (never inspected) | 3,878 |
| Inspections | Missing zip code | 3,108 |
| Inspections | Latitude/longitude of 0 (not a real location) | 3,047 |
| Inspections | Exact duplicate rows | 192 |
| Inspections | Borough listed as 0 | 349 |
| Inspections | critical_flag "Not Applicable" | 8,659 |
| Income | Value -666666666 (Census code for "not available") | 3,153 |
| Income | Value 250,001 (top cap, "$250,000 or more") | 70 |

The 8,659 "Not Applicable" rows are three different things: 3,878 placeholder rows, 2,951 real inspections with no violation found, and 1,830 posting-rule violations (calorie, smoke-free air, trans fat, sodium). The cleaning log shows 1,824 posting-rule rows removed instead of 1,830 because 6 of them were also exact duplicates, already removed in step 2.

### Problems checked for and not found
- Zip codes that are not 5 digits: 0
- Negative scores: 0 (lowest score is 0)
- Income values that are not numbers: 0
- Repeated zip areas in the income data: 0 (33,774 rows, 33,774 distinct areas)
- Inspection dates in the future: 0 (latest real date is 2026-10-07)

## Data dictionary

| Variable | Type | Units / levels | Notes and flags |
|---|---|---|---|
| camis | text (ID) | one ID per restaurant | 31,509 distinct in raw |
| boro | text | Manhattan, Brooklyn, Queens, Bronx, Staten Island | Contradicts documentation: 349 raw rows are "0", which is not a borough. Set to missing. |
| zipcode | text | 5-digit ZIP code | 3,108 missing in raw (2,947 after cleaning, from 362 restaurants). All present values are 5 digits. |
| inspection_date | date | calendar date | Contradicts documentation: 3,878 raw rows are 1900-01-01, a placeholder for "not inspected". Real dates run 2007-08-10 to 2026-10-07. |
| violation_code | text | city code, e.g. 02B | Blank when no violation was found (not an error). 147 distinct. |
| critical_flag | text | Critical, Not Critical, Not Applicable | Not Applicable = placeholder rows, clean inspections, or posting-rule violations. |
| critical_violation (new) | 0 / 1 | 1 = Critical | Added in cleaning. 0 includes inspections with no violation. |
| score | number | violation points, 0 to 214 | Repeats on every violation row of the same inspection, so it is counted once per inspection in the tidy tables. 17,464 missing in raw. |
| latitude, longitude | number | decimal degrees | Contradicts documentation: 3,047 raw rows have 0, which is not in NYC. Set to missing. |
| zcta | text | 5-digit Census ZIP area | An approximation of zip codes, not identical to them. |
| median_household_income | number | US dollars (2022 inflation-adjusted, ACS 2018-2022 estimate) | Contradicts documentation: 3,153 raw values are -666666666, a code, not an income. Set to missing. Real values run $2,499 to $250,001. |
| income_topcoded (new) | True / False | True = value is the $250,001 cap | 70 rows. The cap means "$250,000 or more". |

## 3. Cleaning
The cleaning log below lists every decision with what changed, how many rows or cells, why, and what information is lost. The raw files are never modified.

### Alternatives considered
- **Placeholder rows:** I could have kept them flagged as "never inspected". I removed them because they are not inspections and would add zero-violation rows that are not real.
- **Posting-rule violations (calorie, smoke-free air, trans fat, sodium):** I could have kept them as "non-critical". That would dilute the critical-violation rate with violations unrelated to food safety, so I removed them.
- **Missing zip codes:** I could have looked up locations with a mapping service. Only about 1% of rows are affected, so I kept those rows and they will be skipped at the join in P3.
- **Fake zeros in latitude, longitude and borough:** I set them to missing instead of deleting the rows, because those rows are real inspections.
- **"Not Applicable" on clean inspections:** I kept these 2,951 rows. Dropping them would remove restaurants that passed with no violations and make every area look worse than it is.

In [8]:
import pandas as pd, os
os.makedirs("clean", exist_ok=True)

raw = pd.read_csv("raw/inspections_raw.csv", dtype=str)
df = raw.copy()
log = []
def add(step, what, n, why, lost):
    log.append({"step": step, "what changed": what, "rows/cells affected": n,
                "why": why, "information lost": lost})
    print(f"{step}: {n:,} affected | rows now: {len(df):,}")

# 1. placeholder dates (restaurants never inspected)
is_ph = pd.to_datetime(df.inspection_date, errors="coerce").dt.year == 1900
n = int(is_ph.sum()); df = df[~is_ph]
add("1", "Removed rows with placeholder inspection date 1900-01-01", n,
    "These restaurants were never inspected, so there is no inspection to analyze. They cannot be repaired.",
    "Which restaurants had no inspection yet (still in the raw file)")

# 2. exact duplicate rows
n = int(df.duplicated().sum()); df = df.drop_duplicates()
add("2", "Removed exact duplicate rows (kept first copy)", n,
    "A fully identical row is the same violation recorded twice and would be double counted.",
    "Nothing, the copy kept is identical")

# 3. non-food-safety rows
is_admin = (df.critical_flag == "Not Applicable") & df.violation_code.notna()
n = int(is_admin.sum()); df = df[~is_admin]
add("3", "Removed rows whose violation is a posting rule (calorie, smoke-free air, trans fat, sodium) with no critical flag", n,
    "The question is about food-safety violations, and these rows carry no critical/not critical rating.",
    "Posting-rule violations (still in the raw file)")

# 4. types, with a check that nothing was silently turned into missing
before = df[["inspection_date", "score", "latitude", "longitude"]].isna().sum()
df["inspection_date"] = pd.to_datetime(df.inspection_date, errors="coerce")
for c in ["score", "latitude", "longitude"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
after = df[["inspection_date", "score", "latitude", "longitude"]].isna().sum()
assert (before == after).all(), "type conversion silently created missing values"
add("4", "Converted inspection_date to date and score, latitude, longitude to numbers", 0,
    "Columns were loaded as text to keep the raw values exact. Checked that conversion created no new blanks.",
    "Nothing")

# 5. impossible zero coordinates -> missing
bad = (df.latitude == 0) | (df.longitude == 0)
n = int(bad.sum()); df.loc[bad, ["latitude", "longitude"]] = float("nan")
add("5", "Set latitude and longitude to missing where either was 0", n,
    "0,0 is a point in the ocean, a placeholder for unknown. Rows kept because location is not needed for the zip join.",
    "Nothing real, the 0 was never a location")

# helper: fill a blank from the same restaurant's other rows, only if unambiguous
def fill_from_restaurant(col):
    uniq = df.dropna(subset=[col]).groupby("camis")[col].nunique()
    single = uniq[uniq == 1].index
    lookup = (df[df.camis.isin(single)].dropna(subset=[col])
                .drop_duplicates("camis").set_index("camis")[col])
    mask = df[col].isna() & df.camis.isin(lookup.index)
    df.loc[mask, col] = df.loc[mask, "camis"].map(lookup)
    return int(mask.sum())

# 6. borough 0 -> missing, then repair
z = df.boro == "0"
n = int(z.sum()); df.loc[z, "boro"] = pd.NA
r = fill_from_restaurant("boro")
add("6", f"Borough '0' set to missing, then {r:,} filled from the same restaurant's other rows (only when it had one borough)", n,
     "NYC has 5 boroughs, so 0 is a placeholder. I tried to repair it from the same restaurant's other rows, but none had a usable value, so the rows were kept with borough missing.",
    f"{n - r:,} rows still have no borough")

# 7. repair missing zip codes (the join key)
n = int(df.zipcode.isna().sum())
r = fill_from_restaurant("zipcode")
add("7", f"Filled {r:,} missing zip codes from the same restaurant's other rows (only when it had one zip)", n,
           "Zip code is the key for joining income. I tried to repair it from the same restaurant's other rows, but none had one, so the rows were kept and will be skipped at the join in P3.",
    f"{n - r:,} rows still have no zip and cannot join to income (kept, skipped at the join in P3)")

# 8. new clear column
df["critical_violation"] = (df.critical_flag == "Critical").astype(int)
add("8", "Added critical_violation column: 1 if flag is Critical, 0 if Not Critical or no violation found", len(df),
    "Gives a clean yes/no for the analysis. The 'Not Applicable' rows left are inspections with no violations.",
    "Nothing, original critical_flag column kept")

logdf = pd.DataFrame(log)
logdf.to_csv("clean/cleaning_log.csv", index=False)
df.to_csv("clean/inspections_clean.csv", index=False)
print("\nBEFORE:", raw.shape, " AFTER:", df.shape)
logdf

1: 3,878 affected | rows now: 292,629
2: 192 affected | rows now: 292,437
3: 1,824 affected | rows now: 290,613
4: 0 affected | rows now: 290,613
5: 2,936 affected | rows now: 290,613
6: 281 affected | rows now: 290,613
7: 2,947 affected | rows now: 290,613
8: 290,613 affected | rows now: 290,613

BEFORE: (296507, 27)  AFTER: (290613, 28)


,step,what changed,rows/cells affected,why,information lost
0,1,Removed rows with placeholder inspection date ...,3878,"These restaurants were never inspected, so the...",Which restaurants had no inspection yet (still...
1,2,Removed exact duplicate rows (kept first copy),192,A fully identical row is the same violation re...,"Nothing, the copy kept is identical"
2,3,Removed rows whose violation is a posting rule...,1824,"The question is about food-safety violations, ...",Posting-rule violations (still in the raw file)
3,4,"Converted inspection_date to date and score, l...",0,Columns were loaded as text to keep the raw va...,Nothing
4,5,Set latitude and longitude to missing where ei...,2936,"0,0 is a point in the ocean, a placeholder for...","Nothing real, the 0 was never a location"
5,6,"Borough '0' set to missing, then 0 filled from...",281,"NYC has 5 boroughs, so 0 is a placeholder. I t...",281 rows still have no borough
6,7,Filled 0 missing zip codes from the same resta...,2947,Zip code is the key for joining income. I trie...,"2,947 rows still have no zip and cannot join t..."
7,8,Added critical_violation column: 1 if flag is ...,290613,Gives a clean yes/no for the analysis. The 'No...,"Nothing, original critical_flag column kept"


In [9]:
# --- check: can missing zips be repaired from the same restaurant? ---
miss = df[df.zipcode.isna()]
has_zip = df.groupby("camis").zipcode.apply(lambda s: s.notna().any())
print("restaurants with a missing-zip row:", miss.camis.nunique())
print("...of those, with a zip on some other row:", int(has_zip.loc[miss.camis.unique()].sum()))
print()

# --- clean the income data ---
inc_raw = pd.read_csv("raw/census_income_raw.csv", dtype=str)
inc = inc_raw.rename(columns={"zip code tabulation area": "zcta",
                              "S1901_C01_012E": "median_household_income"})
inc_log = []
def add2(step, what, n, why, lost):
    inc_log.append({"step": step, "what changed": what, "rows/cells affected": n,
                    "why": why, "information lost": lost})
    print(f"{step}: {n:,} affected | rows now: {len(inc):,}")

# 1. NAME column is just 'ZCTA5 ' + the zip code
assert (inc.NAME == "ZCTA5 " + inc.zcta).all()
inc = inc.drop(columns="NAME")
add2("1", "Dropped NAME column", len(inc),
     "It only repeats the zip code (as 'ZCTA5 xxxxx'), checked on every row.", "Nothing")

# 2. income to number, with check
num = pd.to_numeric(inc.median_household_income, errors="coerce")
assert num.notna().all(), "conversion would create new blanks"
inc["median_household_income"] = num
add2("2", "Converted median_household_income to a number", 0,
     "Loaded as text to keep raw values exact. Checked nothing became blank.", "Nothing")

# 3. -666666666 is Census code for 'not available'
bad = inc.median_household_income < 0
n = int(bad.sum()); inc.loc[bad, "median_household_income"] = float("nan")
add2("3", "Replaced the -666666666 code with missing", n,
     "It means 'not available', not a real income. Left as missing instead of 0 so averages are not dragged down.",
     "Nothing real. These zip areas have no usable income")

# 4. flag the top-code
inc["income_topcoded"] = inc.median_household_income == 250001
add2("4", "Added income_topcoded flag for the 250,001 value", int(inc.income_topcoded.sum()),
     "Census caps the top income at 250,001 ('$250,000 or more'), so those are not exact incomes.",
     "Nothing, values kept as published")

inc_logdf = pd.DataFrame(inc_log)
inc_logdf.to_csv("clean/income_cleaning_log.csv", index=False)
inc.to_csv("clean/census_income_clean.csv", index=False)

# --- sanity check for the P3 join (changes nothing) ---
nyc_zips = set(df.zipcode.dropna())
have = inc[inc.zcta.isin(nyc_zips) & inc.median_household_income.notna()]
print("\nNYC zip codes in the inspection data:", len(nyc_zips))
print("...with a usable income value:", have.zcta.nunique())
print("inspection rows whose zip has a usable income:", int(df.zipcode.isin(have.zcta).sum()), "of", len(df))
inc_logdf

restaurants with a missing-zip row: 362
...of those, with a zip on some other row: 0

1: 33,774 affected | rows now: 33,774
2: 0 affected | rows now: 33,774
3: 3,153 affected | rows now: 33,774
4: 70 affected | rows now: 33,774

NYC zip codes in the inspection data: 238
...with a usable income value: 196
inspection rows whose zip has a usable income: 285052 of 290613


,step,what changed,rows/cells affected,why,information lost
0,1,Dropped NAME column,33774,It only repeats the zip code (as 'ZCTA5 xxxxx'...,Nothing
1,2,Converted median_household_income to a number,0,Loaded as text to keep raw values exact. Check...,Nothing
2,3,Replaced the -666666666 code with missing,3153,"It means 'not available', not a real income. L...",Nothing real. These zip areas have no usable i...
3,4,"Added income_topcoded flag for the 250,001 value",70,"Census caps the top income at 250,001 ('$250,0...","Nothing, values kept as published"


## 4. Tidy structure
The raw table broke the tidy rule **"each type of observational unit forms a table."** Restaurant facts (name, address, zip) and inspection facts (date, score, grade) were repeated on every violation row, so one inspection's score would be counted once per violation. I split it into three tables: restaurants (one row per restaurant), inspections (one row per inspection), and violations (one row per violation found). Row counts were checked to match: 287,662 violation rows plus 2,951 clean inspections equals the 290,613 cleaned rows.

In [10]:
key = ["camis", "inspection_date", "inspection_type"]

# restaurants: one row per restaurant (most recent values if anything changed)
rest_cols = ["camis", "dba", "boro", "building", "street", "zipcode", "phone",
             "cuisine_description", "latitude", "longitude", "community_board",
             "council_district", "census_tract", "bin", "bbl", "nta"]
changed = df.groupby("camis")[["boro", "zipcode", "cuisine_description"]].nunique().gt(1).sum()
print("restaurants whose value changed across rows:\n", changed, "\n")
restaurants = (df.sort_values("inspection_date")[rest_cols]
                 .drop_duplicates("camis", keep="last").reset_index(drop=True))

# inspections: one row per inspection
print("inspections with more than one distinct score:",
      int(df.groupby(key, dropna=False).score.nunique().gt(1).sum()))
inspections = (df.groupby(key, dropna=False)
                 .agg(action=("action", "first"), score=("score", "first"),
                      grade=("grade", "first"), grade_date=("grade_date", "first"),
                      n_violations=("violation_code", "count"),
                      n_critical=("critical_violation", "sum"))
                 .reset_index())

# violations: one row per violation found
violations = df.loc[df.violation_code.notna(),
                    key + ["violation_code", "violation_description",
                           "critical_flag", "critical_violation"]].reset_index(drop=True)

# checks
assert restaurants.camis.is_unique
assert inspections.n_violations.sum() == len(violations)
assert inspections.n_critical.sum() == violations.critical_violation.sum()
print("\nrestaurants:", restaurants.shape, "| inspections:", inspections.shape,
      "| violations:", violations.shape)

restaurants.to_csv("clean/tidy_restaurants.csv", index=False)
inspections.to_csv("clean/tidy_inspections.csv", index=False)
violations.to_csv("clean/tidy_violations.csv", index=False)

# add to the cleaning log
logdf = pd.concat([logdf, pd.DataFrame([{
    "step": "9", "what changed": "Split the single table into restaurants, inspections and violations tables",
    "rows/cells affected": len(df),
    "why": "Tidy rule: each type of observational unit gets its own table. The raw table repeated restaurant and inspection facts on every violation row, so a score was counted once per violation.",
    "information lost": "Nothing. Row counts are checked to match."}])], ignore_index=True)
logdf.to_csv("clean/cleaning_log.csv", index=False)
inspections.head()

restaurants whose value changed across rows:
 boro                   0
zipcode                0
cuisine_description    0
dtype: int64 

inspections with more than one distinct score: 0

restaurants: (27631, 16) | inspections: (91863, 9) | violations: (287662, 7)


,camis,inspection_date,inspection_type,action,score,grade,grade_date,n_violations,n_critical
0,30075445,2023-08-01,Cycle Inspection / Initial Inspection,Violations were cited in the following area(s).,38.0,None,None,3,2
1,30075445,2023-08-22,Cycle Inspection / Re-inspection,Violations were cited in the following area(s).,12.0,A,2023-08-22T00:00:00.000,3,1
2,30075445,2024-11-08,Cycle Inspection / Initial Inspection,Violations were cited in the following area(s).,10.0,A,2024-11-08T00:00:00.000,3,1
3,30075445,2026-02-27,Cycle Inspection / Initial Inspection,Violations were cited in the following area(s).,7.0,A,2026-02-27T00:00:00.000,2,1
4,30191841,2024-11-20,Cycle Inspection / Initial Inspection,Violations were cited in the following area(s).,24.0,None,None,6,2


## Self-Scored Rubric

| Criterion | Wt | Score | Why |
|---|---|---|---|
| Diagnosis & data dictionary | ×2 | 3/4 | Another week would add type and range checks for every column and document all 30 variables, not just the 12 most important. |
| Cleaning execution | ×2 | 4/4 | Fixes are counted, alternatives are written down, and nothing repairable was dropped. |
| Cleaning log | ×3 | 4/4 | Every decision has a count, a reason, and what is lost. |
| Provenance Brief (under 200 words) | ×1 | 4/4 | 149 words, no technical terms. |
| Tidy structure & reproducibility | ×2 | 4/4 | Split into three tables, README included, notebook runs top to bottom. |

## AI Use

I used Claude throughout this milestone.

**What Claude did**
- Helped me work out why the Census API key was being rejected (the key had been pasted twice into my Colab Secret, so it was 81 characters instead of 40).
- Wrote the code for downloading the data, the audit, the cleaning steps and the tidy tables.
- Explained each result in plain language, and suggested the cleaning decisions (for example, removing the placeholder rows and keeping the clean inspections with no violations).
- Drafted the data dictionary, the Provenance Brief, the alternatives-considered list, the README and the wording of my self-score.

**What I did**
- Set up Colab, got the Census key, and ran every cell myself.
- Sent the outputs back and checked them against the written text.
- Assembled the notebook and made the final edits.
- Found several errors in Claudes outputs and resolved them.

**What Claude got wrong**
- Claude could not reach the NYC or Census websites from its own environment, so it tested its code only on made-up practice data. The real checks happened when I ran the cells in Colab.
- Claude predicted about 84,000 inspections after tidying. The real run gave 91,863, because the final count splits same-day inspections by type.
- Claude's first cleaning log said missing boroughs and zip codes were "repaired". The output showed 0 were repaired, so the log wording was corrected to match the code.
- Mislabeled inspection data and misunderstood values.